# Вариант 1

In [15]:
import pandas as pd
import numpy as np

# Практическая работа 5
### Оптимизация и очистка данных

**Цель:** Научиться уменьшать потребление памяти и ускорять обработку данных в pandas, а также выявлять и исправлять проблемы качества данных (дубликаты, пропуски, аномальные значения) на примере «больших» CSV и JSON из интернет-магазина.

### Исходные данные

Файлы заказов интернет-магазина за три месяца:
- `data/orders_2024_01.csv` — январь 2024
- `data/orders_2024_02.csv` — февраль 2024
- `data/orders_2024_03.csv` — март 2024

Столбцы: `order_id`, `user_id`, `order_datetime`, `product_id`, `price`, `quantity`, `status`

### Подготовка: Загрузка исходных данных заказов

In [16]:
# Загрузка исходных файлов заказов без оптимизации типов
orders_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv",
]

df_orders_raw = pd.concat([pd.read_csv(f) for f in orders_files], ignore_index=True)

print("Исходная информация о заказах:")
df_orders_raw.info(memory_usage="deep")
df_orders_raw.head()

Исходная информация о заказах:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        15000 non-null  int64  
 1   user_id         15000 non-null  int64  
 2   order_datetime  15000 non-null  str    
 3   product_id      15000 non-null  int64  
 4   price           15000 non-null  float64
 5   quantity        15000 non-null  int64  
 6   status          15000 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 1.6 MB


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid


### Подготовка: Загрузка с оптимизацией типов данных (dtype, category, parse_dates)

In [17]:
# Подбор оптимальных типов данных:
# order_id, user_id -> int32
# product_id, quantity -> int16
# price -> float32
# status -> category
# order_datetime -> parse_dates
orders_dtypes = {
    "order_id": "int32",
    "user_id": "int32",
    "product_id": "int16",
    "price": "float32",
    "quantity": "int16",
    "status": "category",
}

df_orders_opt = pd.concat(
    [
        pd.read_csv(
            f,
            dtype=orders_dtypes,
            parse_dates=["order_datetime"],
        )
        for f in orders_files
    ],
    ignore_index=True,
)

print("Информация об оптимизированных заказах:")
df_orders_opt.info(memory_usage="deep")
df_orders_opt.head()

Информация об оптимизированных заказах:
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        15000 non-null  int32         
 1   user_id         15000 non-null  int32         
 2   order_datetime  15000 non-null  datetime64[us]
 3   product_id      15000 non-null  int16         
 4   price           15000 non-null  float32       
 5   quantity        15000 non-null  int16         
 6   status          15000 non-null  category      
dtypes: category(1), datetime64[us](1), float32(1), int16(2), int32(2)
memory usage: 366.4 KB


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.150391,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.599609,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.259766,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.330078,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.379883,2,paid


### Задание 1. Посчитать потребление памяти исходных заказов и оптимизированных (с dtype и category)

In [18]:
# Расчёт памяти исходного и оптимизированного датафреймов
memory_raw_bytes = df_orders_raw.memory_usage(deep=True).sum()
memory_opt_bytes = df_orders_opt.memory_usage(deep=True).sum()

memory_raw_mb = memory_raw_bytes / 1024**2
memory_opt_mb = memory_opt_bytes / 1024**2

print(f"Память исходных данных:         {memory_raw_bytes:>10,} байт ({memory_raw_mb:.2f} MB)")
print(f"Память оптимизированных данных: {memory_opt_bytes:>10,} байт ({memory_opt_mb:.2f} MB)")

Память исходных данных:          1,709,156 байт (1.63 MB)
Память оптимизированных данных:    375,232 байт (0.36 MB)


### Задание 2. Оценить процент экономии памяти

In [19]:
# Вычисление абсолютной и процентной экономии памяти
memory_saved_bytes = memory_raw_bytes - memory_opt_bytes
memory_saved_mb = memory_raw_mb - memory_opt_mb
savings_percent = 100 * (1 - memory_opt_bytes / memory_raw_bytes)

print(f"Абсолютная экономия памяти: {memory_saved_bytes:>10,} байт ({memory_saved_mb:.2f} MB)")
print(f"Относительная экономия:     {savings_percent:.2f}%")

Абсолютная экономия памяти:  1,333,924 байт (1.27 MB)
Относительная экономия:     78.05%


### Задание 3. Оформить вывод в виде небольшой таблицы (DataFrame)

In [20]:
# 1. Итоговая сводная таблица
summary_df = pd.DataFrame([
    {
        "Датасет": "Заказы (2024_01 - 2024_03)",
        "Строк": len(df_orders_raw),
        "Память ДО (MB)": round(memory_raw_mb, 2),
        "Память ПОСЛЕ (MB)": round(memory_opt_mb, 2),
        "Сэкономлено (MB)": round(memory_saved_mb, 2),
        "Экономия (%)": f"{savings_percent:.1f}%",
    }
])

display(summary_df)

# 2. Детализированная таблица по каждому столбцу
col_records = []
for col in df_orders_raw.columns:
    raw_col = df_orders_raw[col].memory_usage(deep=True)
    opt_col = df_orders_opt[col].memory_usage(deep=True)
    savings_col = (1 - opt_col / raw_col) * 100
    col_records.append({
        "Столбец": col,
        "Исходный dtype": str(df_orders_raw[col].dtype),
        "Оптимизированный dtype": str(df_orders_opt[col].dtype),
        "Память ДО (KB)": round(raw_col / 1024, 2),
        "Память ПОСЛЕ (KB)": round(opt_col / 1024, 2),
        "Экономия (%)": f"{savings_col:.1f}%",
    })

col_summary_df = pd.DataFrame(col_records)
display(col_summary_df)

,Датасет,Строк,Память ДО (MB),Память ПОСЛЕ (MB),Сэкономлено (MB),Экономия (%)
0,Заказы (2024_01 - 2024_03),15000,1.63,0.36,1.27,78.0%


,Столбец,Исходный dtype,Оптимизированный dtype,Память ДО (KB),Память ПОСЛЕ (KB),Экономия (%)
0,order_id,int64,int32,117.26,58.66,50.0%
1,user_id,int64,int32,117.26,58.66,50.0%
2,order_datetime,str,datetime64[us],644.60,117.26,81.8%
3,product_id,int64,int16,117.26,29.37,75.0%
4,price,float64,float32,117.26,58.66,50.0%
5,quantity,int64,int16,117.26,29.37,75.0%
6,status,str,category,438.63,14.88,96.6%


### Очистка данных: проверка на дубликаты и пропуски

In [21]:
# Поиск полных дубликатов по бизнес-ключам
duplicates = df_orders_opt.duplicated(
    subset=["order_id", "user_id", "product_id", "order_datetime"]
)
print("Количество полных дубликатов в заказах:", duplicates.sum())

print("\nКоличество явных пропусков (isna) по столбцам:")
print(df_orders_opt.isna().sum())

print("\nРаспределение категорий в столбце status:")
print(df_orders_opt["status"].value_counts(dropna=False))

Количество полных дубликатов в заказах: 0

Количество явных пропусков (isna) по столбцам:
order_id          0
user_id           0
order_datetime    0
product_id        0
price             0
quantity          0
status            0
dtype: int64

Распределение категорий в столбце status:
status
paid         12032
cancelled     2212
refunded       756
Name: count, dtype: int64
